# Tools y Skills · 2.1.4

Una Tool opera; una Skill declara capacidades e instrucciones. El grafo decide qué se ejecuta. load_skill necesita los recursos locales enlazados, no Studio.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
class State(pydantic.BaseModel):
    value: int

@toolkit.tool
def increment(value: int) -> dict:
    return {'value': value + 1}
graph = toolkit.Graph(State)
graph.add_node('increment', increment)
graph.add_edge(toolkit.START, 'increment')
graph.add_edge('increment', toolkit.END)
agent = toolkit.Agent('counter', graph=graph)


In [4]:
capability = toolkit.skill(name='arithmetic', tools=(increment,), instructions='Use integer arithmetic.')
worker = toolkit.Agent('worker', graph=graph, skills=(capability,))
result = worker.run({'value': 3})
assert result.ok and result.data == {'value': 4}
print(result.human_results())
print(capability.inspect())


worker: COMPLETED
Execution: run-23aaf892eebf4c9285ac7e0c254976eb
Input: {"value":3}
Output: {"value":4}
Process:
- Tool increment: completed -> {"value":4}
{'name': 'arithmetic', 'description': '', 'version': '0.1.0', 'instructions': 'Use integer arithmetic.', 'tools': [{'name': 'increment', 'description': '', 'metadata': {}, 'input_schema': None, 'output_schema': None}], 'metadata': {}}


Carga portable: [carpeta completa](../../examples/portable_skills/assets/arithmetic). SKILL.md documenta; skill.py construye la Skill.

In [5]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/common_graph/application.py').is_file()))
sys.path.insert(0, str(ROOT))
loaded = toolkit.load_skill(ROOT / 'examples/portable_skills/assets/arithmetic')
loaded_result = loaded.tools[0].run({'value': 3})
assert loaded_result.ok and loaded_result.data == {'value': 4}
print(loaded.inspect())


{'name': 'arithmetic', 'description': 'Portable local arithmetic', 'version': '1.0.0', 'instructions': 'Use exact integer arithmetic.', 'tools': [{'name': 'increment', 'description': 'Increment an integer using a sibling resource.', 'metadata': {}, 'input_schema': None, 'output_schema': None}], 'metadata': {'synthetic': True}}
